<a href="https://colab.research.google.com/github/DeepFluxion/Mack_2026_Language_Analitycs/blob/main/notebooks/sessao_2_chart_visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Visualização de Gráficos — pandas 3.0.3

> **Nota:** Os exemplos abaixo assumem que você está usando o Jupyter.

Esta seção demonstra visualização por meio de gráficos. O pandas fornece o básico para criar gráficos de aparência adequada facilmente. Para bibliotecas de visualização que vão além do básico documentado aqui, consulte a página do ecossistema.

## Conteúdo
1. [Plotagem básica: `plot`](#1-plotagem-básica-plot)
2. [Outros tipos de gráfico](#2-outros-tipos-de-gráfico)
3. [Gráficos de Barras](#3-gráficos-de-barras-bar-plots)
4. [Histogramas](#4-histogramas)
5. [Box Plots](#5-box-plots)
6. [Gráfico de Área](#6-gráfico-de-área-area-plot)
7. [Gráfico de Dispersão](#7-gráfico-de-dispersão-scatter-plot)
8. [Hexbin Plot](#8-hexbin-plot)
9. [Gráfico de Pizza](#9-gráfico-de-pizza-pie-plot)
10. [Plotagem com dados ausentes](#10-plotagem-com-dados-ausentes)
11. [Ferramentas de plotagem (`pandas.plotting`)](#11-ferramentas-de-plotagem-pandasplotting)
12. [Formatação de gráficos](#12-formatação-de-gráficos)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Exibe gráficos diretamente no notebook
%matplotlib inline

# Semente para reprodutibilidade (usada em toda a documentação)
np.random.seed(123456)

# Fecha todas as figuras abertas anteriormente
plt.close("all")

---
## 1. Plotagem básica: `plot`

O método `plot` em `Series` e `DataFrame` é apenas um wrapper simples em torno de `plt.plot()`.

### `Series.plot()`

In [ ]:
# Cria uma Series de caminhada aleatória com índice de datas
ts = pd.Series(np.random.randn(1000), index=pd.date_range("1/1/2000", periods=1000))
ts = ts.cumsum()

# plot() usa o índice como eixo X; se for datas, formata automaticamente
ts.plot();

Se o índice consiste em datas, o pandas chama `gcf().autofmt_xdate()` para formatar o eixo X automaticamente.

### `DataFrame.plot()`

Em um `DataFrame`, `plot()` plota todas as colunas com rótulos de uma só vez:

In [ ]:
# DataFrame de 4 colunas com caminhada aleatória
df = pd.DataFrame(np.random.randn(1000, 4), index=ts.index, columns=list("ABCD"))
df = df.cumsum()

plt.figure()
df.plot();

### Plotando uma coluna versus outra

Use os argumentos `x` e `y` para plotar uma coluna contra outra:

In [ ]:
df3 = pd.DataFrame(np.random.randn(1000, 2), columns=["B", "C"]).cumsum()
df3["A"] = pd.Series(list(range(len(df))))

# Plota coluna 'B' no eixo Y versus coluna 'A' no eixo X
df3.plot(x="A", y="B");

---
## 2. Outros tipos de gráfico

Os métodos de plotagem suportam vários estilos além do gráfico de linha padrão. Eles podem ser fornecidos via argumento `kind` para `plot()`, ou acessados diretamente como `DataFrame.plot.<kind>`:

| `kind` | Tipo de gráfico |
|---|---|
| `'bar'` / `'barh'` | Gráfico de barras (vertical / horizontal) |
| `'hist'` | Histograma |
| `'box'` | Boxplot |
| `'kde'` / `'density'` | Gráfico de densidade |
| `'area'` | Gráfico de área |
| `'scatter'` | Gráfico de dispersão |
| `'hexbin'` | Gráfico hexbin |
| `'pie'` | Gráfico de pizza |

In [ ]:
# Exemplo: gráfico de barras via kind='bar'
plt.figure()
df.iloc[5].plot(kind="bar");

Além desses tipos, também existem os métodos `DataFrame.hist()` e `DataFrame.boxplot()`, que usam uma interface separada. Por fim, em `pandas.plotting` há diversas funções que aceitam `Series` ou `DataFrame` como argumento:

- Scatter Matrix
- Andrews Curves
- Parallel Coordinates
- Lag Plot
- Autocorrelation Plot
- Bootstrap Plot
- RadViz

---
## 3. Gráficos de Barras (Bar Plots)

Para dados rotulados que não são séries temporais, você pode criar um gráfico de barras:

In [ ]:
# Gráfico de barras de uma linha (Series) do DataFrame
plt.figure()
df.iloc[5].plot.bar()
plt.axhline(0, color="k");  # linha horizontal em y=0

In [ ]:
# DataFrame com 10 linhas e 4 colunas para demonstrar barras múltiplas
df2 = pd.DataFrame(np.random.rand(10, 4), columns=["a", "b", "c", "d"])

# plot.bar() em um DataFrame gera um gráfico de barras múltiplas
df2.plot.bar();

In [ ]:
# stacked=True: barras empilhadas
df2.plot.bar(stacked=True);

In [ ]:
# barh: barras horizontais empilhadas
df2.plot.barh(stacked=True);

---
## 4. Histogramas

Histogramas podem ser desenhados com `DataFrame.plot.hist()` e `Series.plot.hist()`:

In [ ]:
# DataFrame com 3 distribuições normais centradas em valores diferentes
df4 = pd.DataFrame(
    {
        "a": np.random.randn(1000) + 1,   # centrada em +1
        "b": np.random.randn(1000),        # centrada em 0
        "c": np.random.randn(1000) - 1,   # centrada em -1
    },
    columns=["a", "b", "c"],
)

plt.figure()
# alpha=0.5: transparência para visualizar sobreposição
df4.plot.hist(alpha=0.5);

In [ ]:
plt.figure()
# stacked=True: histograma empilhado; bins=20: número de intervalos
df4.plot.hist(stacked=True, bins=20);

In [ ]:
plt.figure()
# orientation='horizontal': histograma horizontal
# cumulative=True: histograma cumulativo
df4["a"].plot.hist(orientation="horizontal", cumulative=True);

In [ ]:
# Interface clássica DataFrame.hist() ainda pode ser usada
plt.figure()
df["A"].diff().hist();

In [ ]:
plt.figure()
# DataFrame.hist() plota histogramas de cada coluna em subplots separados
df.diff().hist(color="k", alpha=0.5, bins=50);

In [ ]:
# by: histogramas agrupados por uma variável categórica
data = pd.Series(np.random.randn(1000))
data.hist(by=np.random.randint(0, 4, 1000), figsize=(6, 4));

In [ ]:
# by também funciona com DataFrame.plot.hist()
data = pd.DataFrame(
    {
        "a": np.random.choice(["x", "y", "z"], 1000),
        "b": np.random.choice(["e", "f", "g"], 1000),
        "c": np.random.randn(1000),
        "d": np.random.randn(1000) - 1,
    }
)

# Histogramas de 'c' e 'd' agrupados pelas combinações de 'a' e 'b'
data.plot.hist(by=["a", "b"], figsize=(10, 5));

---
## 5. Box Plots

Boxplots podem ser desenhados com `Series.plot.box()`, `DataFrame.plot.box()` ou `DataFrame.boxplot()` para visualizar a distribuição de valores dentro de cada coluna.

Aqui está um boxplot representando cinco ensaios de 10 observações de uma variável aleatória uniforme em `[0, 1)`:

In [ ]:
# Boxplot básico: 10 observações para cada uma das 5 variáveis (A-E)
df = pd.DataFrame(np.random.rand(10, 5), columns=["A", "B", "C", "D", "E"])
df.plot.box();

In [ ]:
# Colorização do boxplot via dict com chaves: boxes, whiskers, medians, caps
color = {
    "boxes": "DarkGreen",
    "whiskers": "DarkOrange",
    "medians": "DarkBlue",
    "caps": "Gray",
}
# sym='r+': estilo dos outliers (fliers) em vermelho
df.plot.box(color=color, sym="r+");

In [ ]:
# vert=False: boxplot horizontal
# positions: posições customizadas para cada caixa no eixo
df.plot.box(vert=False, positions=[1, 4, 5, 6, 8]);

In [ ]:
# Interface clássica DataFrame.boxplot() ainda pode ser usada
df = pd.DataFrame(np.random.rand(10, 5))
plt.figure()
bp = df.boxplot()

In [ ]:
# Boxplot estratificado: coluna 'X' define os grupos (A e B)
df = pd.DataFrame(np.random.rand(10, 2), columns=["Col1", "Col2"])
df["X"] = pd.Series(["A", "A", "A", "A", "A", "B", "B", "B", "B", "B"])

plt.figure()
bp = df.boxplot(by="X")

In [ ]:
# Subconjunto de colunas e agrupamento por múltiplas colunas
df = pd.DataFrame(np.random.rand(10, 3), columns=["Col1", "Col2", "Col3"])
df["X"] = pd.Series(["A", "A", "A", "A", "A", "B", "B", "B", "B", "B"])
df["Y"] = pd.Series(["A", "B", "A", "B", "A", "B", "A", "B", "A", "B"])

plt.figure()
# column: seleciona colunas para plotar; by: agrupa por combinações de X e Y
bp = df.boxplot(column=["Col1", "Col2"], by=["X", "Y"])

In [ ]:
# Agrupamento também funciona com DataFrame.plot.box()
df = pd.DataFrame(np.random.rand(10, 3), columns=["Col1", "Col2", "Col3"])
df["X"] = pd.Series(["A", "A", "A", "A", "A", "B", "B", "B", "B", "B"])

plt.figure()
bp = df.plot.box(column=["Col1", "Col2"], by="X")

### `return_type` no Boxplot

Em `boxplot`, o tipo de retorno é controlado pelo argumento `return_type`. As opções válidas são `{"axes", "dict", "both", None}`:

| `return_type` | Facetado (`by`) | Tipo de saída |
|---|---|---|
| `None` | Não | `axes` |
| `None` | Sim | ndarray 2D de axes |
| `'axes'` | Não | `axes` |
| `'axes'` | Sim | `Series` de axes |
| `'dict'` | Não | dict de artistas |
| `'dict'` | Sim | `Series` de dicts de artistas |
| `'both'` | Não | namedtuple |
| `'both'` | Sim | `Series` de namedtuples |

`Groupby.boxplot` sempre retorna uma `Series` do `return_type`.

In [ ]:
np.random.seed(1234)
df_box = pd.DataFrame(np.random.randn(50, 2))
df_box["g"] = np.random.choice(["A", "B"], size=50)
df_box.loc[df_box["g"] == "B", 1] += 3

# Subplots divididos primeiro por colunas numéricas, depois pelo valor de 'g'
bp = df_box.boxplot(by="g")

In [ ]:
# groupby().boxplot(): subplots divididos primeiro por 'g', depois por colunas numéricas
bp = df_box.groupby("g").boxplot()

---
## 6. Gráfico de Área (Area Plot)

Gráficos de área podem ser criados com `Series.plot.area()` e `DataFrame.plot.area()`. Por padrão são empilhados. Para o gráfico empilhado, cada coluna deve ser totalmente positiva ou totalmente negativa.

> Quando os dados contêm `NaN`, eles são automaticamente preenchidos com `0`. Se desejar descartar ou preencher com valores diferentes, use `dropna()` ou `fillna()` antes de plotar.

In [ ]:
df = pd.DataFrame(np.random.rand(10, 4), columns=["a", "b", "c", "d"])

# Gráfico de área empilhado (padrão)
df.plot.area();

In [ ]:
# stacked=False: gráfico de área não-empilhado
# alpha=0.5 é aplicado automaticamente quando não especificado
df.plot.area(stacked=False);

---
## 7. Gráfico de Dispersão (Scatter Plot)

Gráficos de dispersão podem ser desenhados com `DataFrame.plot.scatter()`. Requer colunas numéricas para os eixos X e Y, especificadas pelos argumentos `x` e `y`:

In [ ]:
df = pd.DataFrame(np.random.rand(50, 4), columns=["a", "b", "c", "d"])
df["species"] = pd.Categorical(
    ["setosa"] * 20 + ["versicolor"] * 20 + ["virginica"] * 10
)

# Scatter básico: coluna 'a' no eixo X e 'b' no eixo Y
df.plot.scatter(x="a", y="b");

In [ ]:
# Múltiplos grupos no mesmo eixo: repete plot.scatter() com ax compartilhado
ax = df.plot.scatter(x="a", y="b", color="DarkBlue", label="Grupo 1")
df.plot.scatter(x="c", y="d", color="DarkGreen", label="Grupo 2", ax=ax);

In [ ]:
# c: nome de coluna numérica para colorir cada ponto (colormap contínuo)
df.plot.scatter(x="a", y="b", c="c", s=50);

In [ ]:
# c com coluna categórica: produz colorbar discreto
df.plot.scatter(x="a", y="b", c="species", cmap="viridis", s=50);

In [ ]:
# Bubble chart: s define o tamanho de cada bolha a partir de uma coluna
df.plot.scatter(x="a", y="b", s=df["c"] * 200);

---
## 8. Hexbin Plot

Hexbin plots podem ser criados com `DataFrame.plot.hexbin()`. São uma alternativa útil a gráficos de dispersão quando os dados são densos demais para plotar cada ponto individualmente.

O argumento `gridsize` controla o número de hexágonos na direção X (padrão: `100`). Um `gridsize` maior significa bins menores e mais numerosos.

In [ ]:
df = pd.DataFrame(np.random.randn(1000, 2), columns=["a", "b"])
df["b"] = df["b"] + np.arange(1000)

# hexbin básico: conta o número de pontos em cada célula hexagonal
df.plot.hexbin(x="a", y="b", gridsize=25);

In [ ]:
# C: valor em cada ponto (x, y); reduce_C_function: agrega os valores do bin
# Aqui: posições de 'a' e 'b', valor de 'z', agregado pelo máximo
df = pd.DataFrame(np.random.randn(1000, 2), columns=["a", "b"])
df["b"] = df["b"] + np.arange(1000)
df["z"] = np.random.uniform(0, 3, 1000)

df.plot.hexbin(x="a", y="b", C="z", reduce_C_function=np.max, gridsize=25);

---
## 9. Gráfico de Pizza (Pie Plot)

Gráficos de pizza podem ser criados com `DataFrame.plot.pie()` ou `Series.plot.pie()`. Se os dados contiverem `NaN`, serão automaticamente substituídos por `0`. Um `ValueError` será levantado se houver valores negativos.

> Para gráficos de pizza, é melhor usar figuras quadradas (aspect ratio = 1).

In [ ]:
# Gráfico de pizza a partir de uma Series
series = pd.Series(3 * np.random.rand(4), index=["a", "b", "c", "d"], name="serie")
series.plot.pie(figsize=(6, 6));

In [ ]:
# DataFrame.plot.pie() requer y= (coluna específica) ou subplots=True
df = pd.DataFrame(
    3 * np.random.rand(4, 2), index=["a", "b", "c", "d"], columns=["x", "y"]
)

# subplots=True: gráfico de pizza para cada coluna como subplot
df.plot.pie(subplots=True, figsize=(8, 4));

> **Aviso:** A maioria dos plots do pandas usa `label` e `color` (sem 's'). Para ser consistente com `matplotlib.pyplot.pie()`, nos gráficos de pizza você deve usar `labels` e `colors` (com 's').

In [ ]:
# Customizando rótulos, cores, formato de porcentagem e tamanho de fonte
series.plot.pie(
    labels=["AA", "BB", "CC", "DD"],
    colors=["r", "g", "b", "c"],
    autopct="%.2f",   # exibe porcentagem com 2 casas decimais
    fontsize=20,
    figsize=(6, 6),
);

In [ ]:
# Valores cuja soma é menor que 1.0 são reescalonados para somar 1
series2 = pd.Series([0.1] * 4, index=["a", "b", "c", "d"], name="series2")
series2.plot.pie(figsize=(6, 6));

---
## 10. Plotagem com dados ausentes

O pandas tenta ser pragmático ao plotar `DataFrames` ou `Series` com dados ausentes. Os valores ausentes são descartados, omitidos ou preenchidos dependendo do tipo de gráfico:

| Tipo de gráfico | Tratamento de NaN |
|---|---|
| Linha | Deixa lacunas nos NaNs |
| Linha (empilhada) | Preenche com `0` |
| Barra | Preenche com `0` |
| Dispersão | Remove NaNs |
| Histograma | Remove NaNs (por coluna) |
| Box | Remove NaNs (por coluna) |
| Área | Preenche com `0` |
| KDE | Remove NaNs (por coluna) |
| Hexbin | Remove NaNs |
| Pizza | Preenche com `0` |

Se algum desses comportamentos padrão não for adequado, use `fillna()` ou `dropna()` antes de plotar.

---
## 11. Ferramentas de plotagem (`pandas.plotting`)

Essas funções podem ser importadas de `pandas.plotting` e aceitam `Series` ou `DataFrame` como argumento.

### 11.1 Scatter Matrix

Você pode criar uma matriz de gráficos de dispersão com `scatter_matrix`:

In [ ]:
from pandas.plotting import scatter_matrix

df = pd.DataFrame(np.random.randn(1000, 4), columns=["a", "b", "c", "d"])

# diagonal='kde': curva de densidade na diagonal principal
# alpha=0.2: pontos semi-transparentes para grandes volumes de dados
scatter_matrix(df, alpha=0.2, figsize=(6, 6), diagonal="kde");

### 11.2 Gráfico de Densidade (KDE)

Gráficos de densidade podem ser criados com `Series.plot.kde()` e `DataFrame.plot.kde()`:

In [ ]:
# KDE (Kernel Density Estimation): estimativa de densidade por kernel
ser = pd.Series(np.random.randn(1000))
ser.plot.kde();

### 11.3 Andrews Curves

As Curvas de Andrews permitem plotar dados multivariados como um grande número de curvas criadas usando os atributos das amostras como coeficientes de séries de Fourier. Colorindo as curvas de forma diferente para cada classe, é possível visualizar agrupamentos nos dados.

> **Nota:** O dataset "Iris" é necessário para este exemplo. Utilizamos `sklearn` para carregá-lo.

In [ ]:
from pandas.plotting import andrews_curves
from sklearn.datasets import load_iris

# Carrega o dataset Iris via sklearn
iris = load_iris()
iris_df = pd.DataFrame(iris.data, columns=iris.feature_names)
iris_df["Name"] = [iris.target_names[i] for i in iris.target]

plt.figure()
# Cada curva é uma amostra; curvas da mesma classe tendem a se agrupar
andrews_curves(iris_df, "Name");

### 11.4 Parallel Coordinates

Coordenadas paralelas é uma técnica de plotagem para dados multivariados. Cada linha vertical representa um atributo; um conjunto de segmentos conectados representa um ponto de dados. Pontos que tendem a se agrupar aparecem mais próximos uns dos outros:

In [ ]:
from pandas.plotting import parallel_coordinates

plt.figure()
parallel_coordinates(iris_df, "Name");

### 11.5 Lag Plot

Lag plots são usados para verificar se um conjunto de dados ou série temporal é aleatório. Dados aleatórios não devem exibir nenhuma estrutura no lag plot. Quando `lag=1`, o gráfico é essencialmente `data[:-1]` vs. `data[1:]`:

In [ ]:
from pandas.plotting import lag_plot

plt.figure()
# Série com componente senoidal: estrutura clara no lag plot
spacing = np.linspace(-99 * np.pi, 99 * np.pi, num=1000)
data = pd.Series(0.1 * np.random.rand(1000) + 0.9 * np.sin(spacing))
lag_plot(data);

### 11.6 Autocorrelation Plot

Gráficos de autocorrelação são usados para verificar aleatoriedade em séries temporais, calculando autocorrelações para diferentes defasagens de tempo. Se a série for aleatória, as autocorrelações devem ser próximas de zero. As linhas horizontais correspondem a bandas de confiança de 95% e 99%:

In [ ]:
from pandas.plotting import autocorrelation_plot

plt.figure()
# Série com componente senoidal: autocorrelações significativamente não-nulas
spacing = np.linspace(-9 * np.pi, 9 * np.pi, num=1000)
data = pd.Series(0.7 * np.random.rand(1000) + 0.3 * np.sin(spacing))
autocorrelation_plot(data);

### 11.7 Bootstrap Plot

Bootstrap plots são usados para avaliar visualmente a incerteza de uma estatística (média, mediana, midrange, etc.). Um subconjunto aleatório de tamanho especificado é selecionado do conjunto de dados, a estatística é calculada, e o processo se repete um número especificado de vezes:

In [ ]:
from pandas.plotting import bootstrap_plot

data = pd.Series(np.random.rand(1000))

# size=50: tamanho de cada subconjunto; samples=500: número de repetições
bootstrap_plot(data, size=50, samples=500, color="grey");

### 11.8 RadViz

RadViz é uma forma de visualizar dados multivariados. Pontos igualmente espaçados em um círculo unitário representam os atributos. Cada amostra é ligada a esses pontos por molas com rigidez proporcional ao valor numérico do atributo. O ponto de equilíbrio das forças indica onde a amostra é desenhada, colorida de acordo com sua classe:

In [ ]:
from pandas.plotting import radviz

plt.figure()
radviz(iris_df, "Name");

---
## 12. Formatação de gráficos

### 12.1 Definindo o estilo do gráfico

O matplotlib oferece uma gama de estilos de plotagem pré-configurados. Definir o estilo é simples:

In [ ]:
import matplotlib

# Lista todos os estilos disponíveis
print(matplotlib.style.available)

In [ ]:
# Para aplicar um estilo: matplotlib.style.use('ggplot')
# Exemplo com estilo padrão — style='k--' aplica linha tracejada preta
plt.figure()
ts.plot(style="k--", label="Series");

Para cada tipo de gráfico (ex.: `line`, `bar`, `scatter`), argumentos adicionais são repassados para a função matplotlib correspondente (`ax.plot()`, `ax.bar()`, `ax.scatter()`), permitindo controle de estilo além do que o pandas oferece.

### 12.2 Controlando a legenda

Use `legend=False` para ocultar a legenda (exibida por padrão):

In [ ]:
df = pd.DataFrame(np.random.randn(1000, 4), index=ts.index, columns=list("ABCD"))
df = df.cumsum()

# legend=False: oculta a legenda do gráfico
df.plot(legend=False);

### 12.3 Controlando os rótulos dos eixos

Use `xlabel` e `ylabel` para definir rótulos customizados. Por padrão, o pandas usa o nome do índice como `xlabel` e deixa o `ylabel` vazio:

In [ ]:
# Padrão: xlabel = nome do índice, ylabel = vazio
df.plot();

In [ ]:
# Rótulos customizados para ambos os eixos
df.plot(xlabel="novo x", ylabel="novo y");

### 12.4 Escalas (Scales)

Use `logy=True` para escala logarítmica no eixo Y. Veja também `logx` e `loglog`:

In [ ]:
# Cria série com crescimento exponencial
ts = pd.Series(np.random.randn(1000), index=pd.date_range("1/1/2000", periods=1000))
ts = np.exp(ts.cumsum())

# logy=True: escala logarítmica no eixo Y
ts.plot(logy=True);

### 12.5 Eixo Y secundário

Use `secondary_y=True` para plotar dados em um eixo Y secundário:

In [ ]:
# Coluna 'A' no eixo Y primário; coluna 'B' no eixo Y secundário
df["A"].plot()
df["B"].plot(secondary_y=True, style="g");

In [ ]:
# secondary_y com lista de colunas; rótulos dos eixos customizados
plt.figure()
ax = df.plot(secondary_y=["A", "B"])
ax.set_ylabel("Escala CD")
ax.right_ax.set_ylabel("Escala AB");

In [ ]:
# mark_right=False: remove a marcação '(right)' automática na legenda
plt.figure()
df.plot(secondary_y=["A", "B"], mark_right=False);

### 12.6 Suprimindo ajuste de resolução de ticks

O pandas inclui ajuste automático de resolução de ticks para dados de séries temporais de frequência regular. Em casos onde o pandas não consegue inferir a informação de frequência, use `x_compat=True` para suprimir este comportamento:

In [ ]:
# Comportamento padrão: rótulos dos ticks do eixo X ajustados automaticamente
plt.figure()
df["A"].plot();

In [ ]:
# x_compat=True: suprime o ajuste automático de resolução
plt.figure()
df["A"].plot(x_compat=True);

In [ ]:
# Para múltiplos plots: use pd.plotting.plot_params.use() como context manager
plt.figure()
with pd.plotting.plot_params.use("x_compat", True):
    df["A"].plot(color="r")
    df["B"].plot(color="g")
    df["C"].plot(color="b")

### 12.7 Subplots

Cada `Series` em um `DataFrame` pode ser plotada em um eixo diferente com o argumento `subplots=True`:

In [ ]:
# subplots=True: cada coluna em seu próprio subplot
df.plot(subplots=True, figsize=(6, 6));

### 12.8 Layout e múltiplos eixos

O layout dos subplots pode ser especificado com o argumento `layout=(linhas, colunas)`. Use `-1` em uma dimensão para calcular automaticamente o número necessário:

In [ ]:
# layout=(2, 3): 2 linhas × 3 colunas; sharex=False: eixos X independentes
df.plot(subplots=True, layout=(2, 3), figsize=(6, 6), sharex=False);

In [ ]:
# layout=(2, -1): 2 linhas; colunas calculadas automaticamente (-1)
# Equivalente ao exemplo acima
df.plot(subplots=True, layout=(2, -1), figsize=(6, 6), sharex=False);

In [ ]:
# Passando eixos pré-criados via ax: permite layouts mais complexos
# df tem 4 colunas (A, B, C, D) → precisamos de 4 eixos
fig, axes = plt.subplots(2, 2, figsize=(8, 6))

# axes.ravel() converte a grade 2D de eixos em array 1D
df.plot(subplots=True, ax=axes.ravel(), sharex=False, sharey=False);
plt.tight_layout()